<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_matching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
men_meta = pd.read_csv('/content/Mens_metadata_12_3.csv')
men_meta = pd.read_csv('/content/Mens_metadata_12_3.csv')
women_meta_crime = pd.read_csv('/content/women_meta_crime.csv')
save_path = '/content/drive/MyDrive/Capital_trials/'
full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')

#Create New matched Groups based on Scores

taking a caliper + NN approach


In [ ]:
df_encoded = pd.read_csv('')

In [ ]:
print(caliper_race_crime)
print(caliper_state_race_crime)

In [ ]:
#race crime and state
X = df_encoded[['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']


model = LogisticRegression(random_state=0)
# model.fit(X, T)

X = X.replace(r'^\s*$', pd.NA, regex=True)  # '' -> NA
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0)

T = pd.to_numeric(T, errors="coerce")
mask = T.notna()
X, T = X.loc[mask], T.loc[mask]

model.fit(X, T.astype(int))


crime_race_state_propensity = model.predict_proba(X)[:, 1]



full_meta['race_crime_state_prop'] = crime_race_state_propensity


#race and crime
X = df_encoded[['Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]]
T = df_encoded['gender_women']


model = LogisticRegression(random_state=0)
# model.fit(X, T)

X = X.replace(r'^\s*$', pd.NA, regex=True)  # '' -> NA
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(0)

T = pd.to_numeric(T, errors="coerce")
mask = T.notna()
X, T = X.loc[mask], T.loc[mask]

model.fit(X, T.astype(int))


crime_race_propensity = model.predict_proba(X)[:, 1]

full_meta['race_crime_prop'] = crime_race_propensity

In [ ]:
#based on geeks for geeks propensty score implementation

#match groups based on propensity, start with race, crime, state group

women_group = df_encoded[df_encoded['gender_women'] == 1].reset_index().rename(columns={"index": "women_idx"})
men_group = df_encoded[df_encoded['gender_women'] == 0].reset_index().rename(columns={"index": "men_idx"})

nn = NearestNeighbors(n_neighbors=1)
nn.fit(men_group[['race_crime_state_prop']])
distances, indices = nn.kneighbors(women_group[['race_crime_state_prop']])

pairs = []
seen_men = set()

for i, (dist, j) in enumerate(zip(distances.ravel(), indices.ravel())):
    m_idx = int(men_group.loc[j, "men_idx"])
    if m_idx in seen_men:  # without replacement
        continue
    if dist > caliper_state_race_crime:
        continue

    seen_men.add(m_idx)
    pairs.append({
        "women_idx": int(women_group.loc[i, "women_idx"]),
        "men_idx": m_idx,
        "women_prop": float(women_group.loc[i, 'race_crime_state_prop']),
        "men_prop": float(men_group.loc[j, 'race_crime_state_prop']),
        "distance": float(dist),
    })

final = pd.DataFrame(pairs)